In [ ]:

!pip install -q torch torchvision torchaudio numpy pandas matplotlib pillow

## Global Constants
Constants defined in main.py that are used by multiple modules.

In [ ]:
connector_cost = 0.165  
epoxy_cost = 0.515 
fixed_cost = connector_cost + epoxy_cost  


## pratt_torch.py

In [ ]:
import torch


DTYPE = torch.float64

def softmin_torch(values, softness=1e-3):
    v = torch.stack(values) if isinstance(values, (list, tuple)) else values
    
    v_finite = v[torch.isfinite(v)]
    if v_finite.numel() == 0:
        return torch.tensor(float('inf'), dtype=v.dtype, device=v.device)
    return -softness * torch.logsumexp(-v_finite / softness, dim=0)


def max_load_torch(angle, height, length,
                   incline_thickness, diagonal_thickness, mid_vert_thickness, side_vert_thickness, top_thickness, bottom_thickness,
                   incline_depth, diagonal_depth, mid_vert_depth, side_vert_depth, top_depth, bottom_depth,
                   E, sigma_compression, sigma_tension,
                   K=1.0,
                   softness=1e-6,
                   dtype=DTYPE,
                   device=None):
    
    def T(x):
        if isinstance(x, torch.Tensor):
            return x.to(dtype=dtype, device=device)
        return torch.tensor(x, dtype=dtype, device=device)

    angle = T(angle)
    height = T(height)
    length = T(length)
    E = T(E)
    sigma_compression = T(sigma_compression)
    sigma_tension = T(sigma_tension)
    K = T(K)

    
    incline_thickness = T(incline_thickness)
    diagonal_thickness = T(diagonal_thickness)
    mid_vert_thickness = T(mid_vert_thickness)
    side_vert_thickness = T(side_vert_thickness)
    top_thickness = T(top_thickness)
    bottom_thickness = T(bottom_thickness)

    incline_depth = T(incline_depth)
    diagonal_depth = T(diagonal_depth)
    mid_vert_depth = T(mid_vert_depth)
    side_vert_depth = T(side_vert_depth)
    top_depth = T(top_depth)
    bottom_depth = T(bottom_depth)

    
    A_incline = incline_thickness * incline_depth
    A_diagonal = diagonal_thickness * diagonal_depth
    A_mid = mid_vert_thickness * mid_vert_depth
    A_side = side_vert_thickness * side_vert_depth
    A_top = top_thickness * top_depth
    A_bottom = bottom_thickness * bottom_depth

    
    
    I_incline_in = incline_depth * incline_thickness**3 / 12.0
    I_diagonal_in = diagonal_depth * diagonal_thickness**3 / 12.0
    I_mid_in = mid_vert_depth * mid_vert_thickness**3 / 12.0
    I_side_in = side_vert_depth * side_vert_thickness**3 / 12.0
    I_top_in = top_depth * top_thickness**3 / 12.0
    I_bottom_in = bottom_depth * bottom_thickness**3 / 12.0

    
    I_incline_out = incline_thickness * incline_depth**3 / 12.0
    I_diagonal_out = diagonal_thickness * diagonal_depth**3 / 12.0
    I_mid_out = mid_vert_thickness * mid_vert_depth**3 / 12.0
    I_side_out = side_vert_thickness * side_vert_depth**3 / 12.0
    I_top_out = top_thickness * top_depth**3 / 12.0
    I_bottom_out = bottom_thickness * bottom_depth**3 / 12.0

    
    phi = torch.atan(((length - 2 * height / torch.tan(angle)) / 2) / (height))

    
    L_incline = height / torch.sin(angle)
    L_diagonal = height / torch.cos(phi)
    L_top = length - 2 * height / torch.tan(angle)
    L_bottom = length
    L_mid_vert = height
    L_side_vert = height

    
    a_incline = 1.0 / (2.0 * torch.sin(angle))
    a_diagonal = 1.0 / (6.0 * torch.cos(phi))
    a_top = a_incline * torch.cos(angle) + a_diagonal * torch.sin(phi)
    a_bottom = a_top

    total_vertical_area = 2.0 * A_side + A_mid
    
    a_mid = (A_mid / total_vertical_area) * (1.0 / 3.0)
    a_side = (A_side / total_vertical_area) * (1.0 / 3.0)

    
    
    
    lever_top = length / 2.0 - height / torch.tan(angle)
    b_top_center = a_side * lever_top
    b_bottom_2 = -a_side * lever_top

    
    failure_loads = {}
    
    
    def tension_Fcrit_axial_only(a_coeff, area, sigma):
        
        a_safe = torch.clamp(torch.abs(a_coeff), min=1e-12)
        return (area * sigma) / a_safe

    def tension_Fcrit_combined(a_coeff, area, b_coeff, c, I, sigma):
        
        denom = torch.clamp(torch.abs(a_coeff) / area + torch.abs(b_coeff) * c / I, min=1e-12)
        return sigma / denom

    def euler_buckling_Fcrit(E, I, K, L, a_coeff, area, b_coeff=0, c=0):
        L_safe = torch.clamp(L, min=1e-12)
        F_cr_member = (torch.pi**2 * E * I) / (K * L_safe)**2
        
        
        a_safe = torch.clamp(torch.abs(a_coeff), min=1e-12)
        
        
        if isinstance(b_coeff, (int, float)) and b_coeff == 0:
            
            return F_cr_member / a_safe
        
        b_abs = torch.abs(b_coeff) if isinstance(b_coeff, torch.Tensor) else torch.abs(torch.tensor(b_coeff, dtype=a_coeff.dtype, device=a_coeff.device))
        
        
        denom = a_safe + b_abs * c * area / I
        denom_safe = torch.clamp(denom, min=1e-12)
        
        return F_cr_member / denom_safe

    def compression_combined_stress_Fcrit(a_coeff, area, b_coeff, c, I, sigma):
        
        denom = torch.clamp(torch.abs(a_coeff) / area + torch.abs(b_coeff) * c / I, min=1e-12)
        return sigma / denom

    def compression_combined_stress_Fcrit_no_moment(a_coeff, area, sigma):
        
        denom = torch.clamp(torch.abs(a_coeff) / area, min=1e-12)
        return sigma / denom

    
    
    failure_loads['diagonal_rupture'] = tension_Fcrit_axial_only(a_diagonal, A_diagonal, sigma_tension)
    
    
    c_bottom = bottom_thickness / 2.0
    failure_loads['bottom_chord_rupture'] = tension_Fcrit_combined(
        a_bottom, A_bottom, b_bottom_2, c_bottom, I_bottom_in, sigma_tension
    )

    
    
    
    
    failure_loads['incline_buckle'] = euler_buckling_Fcrit(E, I_incline_in, K, L_incline, a_incline, A_incline)
    failure_loads['incline_buckle_out_of_plane'] = euler_buckling_Fcrit(E, I_incline_out, K, L_incline, a_incline, A_incline)
    failure_loads['incline_combined_stress'] = compression_combined_stress_Fcrit_no_moment(
        a_incline, A_incline, sigma_compression
    )
    
    
    c_top = top_thickness / 2.0
    failure_loads['top_chord_buckle'] = euler_buckling_Fcrit(E, I_top_in, K, L_top, a_top, A_top, b_top_center, c_top)
    failure_loads['top_chord_buckle_out_of_plane'] = euler_buckling_Fcrit(E, I_top_out, K, L_top, a_top, A_top, b_top_center, c_top)
    failure_loads['top_chord_combined_stress'] = compression_combined_stress_Fcrit(
        a_top, A_top, b_top_center, c_top, I_top_in, sigma_compression
    )
    
    
    failure_loads['mid_vert_buckle'] = euler_buckling_Fcrit(E, I_mid_in, K, L_mid_vert, a_mid, A_mid)
    failure_loads['mid_vert_buckle_out_of_plane'] = euler_buckling_Fcrit(E, I_mid_out, K, L_mid_vert, a_mid, A_mid)
    failure_loads['mid_vert_combined_stress'] = compression_combined_stress_Fcrit_no_moment(
        a_mid, A_mid, sigma_compression
    )
    
    
    failure_loads['side_vert_buckle'] = euler_buckling_Fcrit(E, I_side_in, K, L_side_vert, a_side, A_side)
    failure_loads['side_vert_buckle_out_of_plane'] = euler_buckling_Fcrit(E, I_side_out, K, L_side_vert, a_side, A_side)
    failure_loads['side_vert_combined_stress'] = compression_combined_stress_Fcrit_no_moment(
        a_side, A_side, sigma_compression
    )

    
    vals = list(failure_loads.values())
    
    return softmin_torch(vals, softness=softness), failure_loads



def compute_volume_torch(incline_thickness, diagonal_thickness, mid_vert_thickness, side_vert_thickness,
                         top_thickness, bottom_thickness,
                         incline_depth, diagonal_depth, mid_vert_depth, side_vert_depth, top_depth, bottom_depth,
                         length, height, angle,
                         dtype=DTYPE, device=None):
    
    def T(x):
        if isinstance(x, torch.Tensor):
            return x.to(dtype=dtype, device=device)
        return torch.tensor(x, dtype=dtype, device=device)

    thickness_dict = {
        'incline': T(incline_thickness),
        'diagonal': T(diagonal_thickness),
        'mid_vert': T(mid_vert_thickness),
        'side_vert': T(side_vert_thickness),
        'top_chord': T(top_thickness),
        'bottom_chord': T(bottom_thickness),
    }
    
    depth_dict = {
        'incline': T(incline_depth),
        'diagonal': T(diagonal_depth),
        'mid_vert': T(mid_vert_depth),
        'side_vert': T(side_vert_depth),
        'top_chord': T(top_depth),
        'bottom_chord': T(bottom_depth),
    }
    
    length = T(length)
    height = T(height)
    angle = T(angle)
    
    
    phi = torch.atan(((length - 2*height/torch.tan(angle))/2) / height)
    
    
    phi = torch.atan(((length - 2*height/torch.tan(angle))/2) / height)
    
    
    L_incline = height / torch.sin(angle)  
    L_diagonal = height / torch.cos(phi)  
    L_top = length - 2*height/torch.tan(angle)  
    L_bottom = length  
    L_mid_vert = height  
    L_side_vert = height  
    
    
    volume = (
        2 * thickness_dict['incline'] * depth_dict['incline'] * L_incline +
        2 * thickness_dict['diagonal'] * depth_dict['diagonal'] * L_diagonal +
        thickness_dict['top_chord'] * depth_dict['top_chord'] * L_top +
        thickness_dict['bottom_chord'] * depth_dict['bottom_chord'] * L_bottom +
        thickness_dict['mid_vert'] * depth_dict['mid_vert'] * L_mid_vert +
        2 * thickness_dict['side_vert'] * depth_dict['side_vert'] * L_side_vert
    )
    
    return volume


def compute_weight_torch(incline_thickness, diagonal_thickness, mid_vert_thickness, side_vert_thickness,
                         top_thickness, bottom_thickness,
                         incline_depth, diagonal_depth, mid_vert_depth, side_vert_depth, top_depth, bottom_depth,
                         length, height, angle,
                         density=7850.0,
                         dtype=DTYPE, device=None):
    def T(x):
        if isinstance(x, torch.Tensor):
            return x.to(dtype=dtype, device=device)
        return torch.tensor(x, dtype=dtype, device=device)
    
    volume = compute_volume_torch(
        incline_thickness, diagonal_thickness, mid_vert_thickness, side_vert_thickness,
        top_thickness, bottom_thickness,
        incline_depth, diagonal_depth, mid_vert_depth, side_vert_depth, top_depth, bottom_depth,
        length, height, angle,
        dtype=dtype, device=device
    )
    
    density = T(density)
    g = T(9.81)
    weight = volume * density * g  
    
    return weight

## materials.py

In [ ]:
from dataclasses import dataclass
@dataclass
class Material:
    E: float
    sigma_compression: float
    sigma_tension: float
    density: float
    def __repr__(self):
        return f"{self.__class__.__name__}"

class Steel(Material):
    def __init__(self):
        super().__init__(
            E=200e9,
            sigma_compression=250e6,
            sigma_tension=400e6,
            density=7850.0
        )

class Aluminum(Material):
    def __init__(self):
        super().__init__(
            E=69e9,
            sigma_compression=150e6,
            sigma_tension=200e6,
            density=2700.0
        )

class Titanium(Material):
    def __init__(self):
        super().__init__(
            E=19.6e6,
            sigma_compression=900e6,
            sigma_tension=950e6,
            density=4500.0
        )

class OchromaWood(Material): 
    def __init__(self):
        super().__init__(
            E=3.71e9, 
            sigma_compression=11.6e6, 
            sigma_tension=19.6e6, 
            density=200.0 
        )

class OchromaWithEpoxy(Material): 
    def __init__(self):
        super().__init__(
            E=3.71e9, 
            sigma_compression=11.6e6, 
            sigma_tension=19.6e6, 
            density=630.0 
        )

class ConservativeBalsaWood(Material):
    def __init__(self):
        
        
        E = 2.0e9                  
        sigma_compression = 2.5e6  
        sigma_tension = 3.0e6      
        density = 240              
        super().__init__(E, sigma_compression, sigma_tension, density)

class LowDensityBalsaWood(Material):
    def __init__(self):
        super().__init__(
            E=3e9,
            sigma_compression=10e6,
            sigma_tension=15e6,
            density=160.0
        )

class HighDensityBalsaWood(Material):
    def __init__(self):
        super().__init__(
            E=6e9,
            sigma_compression=20e6,
            sigma_tension=25e6,
            density=320.0
        )

class Concrete(Material):
    def __init__(self):
        super().__init__(
            E=30e9,
            sigma_compression=40e6,
            sigma_tension=4e6,
            density=2400.0
        )

class PolylacticAcid(Material):
    def __init__(self):
        super().__init__(
            E=3.5e9,
            sigma_compression=60e6,
            sigma_tension=50e6,
            density=1250.0
        )

class CarbonFiberPLA(Material):
    def __init__(self):
        super().__init__(
            E=10e9,
            sigma_compression=100e6,
            sigma_tension=90e6,
            density=1350.0
        
        )

class CustomMaterial(Material):
    def __init__(self, E: float, sigma_compression: float, sigma_tension: float, density: float):
        super().__init__(E, sigma_compression, sigma_tension, density)


## utils.py

In [ ]:
def inches_to_meters(inches):
    return inches * 0.0254

def feet_to_meters(feet):
    return feet * 0.3048

import numpy as np
from scipy.optimize import root

def find_root_excluding(f, x_domain, exclude_range: tuple=None, 
                        max_attempts=50, tol=1e-8):

    xmin, xmax = x_domain

    
    guesses = np.linspace(xmin, xmax, max_attempts)

    found_roots = []

    for x0 in guesses:
        sol = root(f, x0)

        if not sol.success:
            continue

        x = sol.x.item()

        
        if not np.isfinite(x):
            continue

        
        if any(abs(x - r) < tol for r in found_roots):
            continue

        
        if exclude_range is not None:
            a, b = exclude_range
            if a < x < b:
                continue

        
        return x

    raise RuntimeError(
        "No root found outside the excluded range after trying all guesses."
    )

## pratt_optimizer.py

In [ ]:

import torch
import math
from typing import Dict, Tuple, Optional
from dataclasses import dataclass, asdict


@dataclass
class BridgeDesignParams:
    

    
    material: Optional[Material] = None

    
    angle: float = math.radians(30)
    height: float = inches_to_meters(6.0)
    length: float = inches_to_meters(18.5)
    
    
    incline_thickness: float = 0.02
    diagonal_thickness: float = 0.02
    mid_vert_thickness: float = 0.02
    side_vert_thickness: float = 0.02
    top_thickness: float = 0.02
    bottom_thickness: float = 0.02
    
    incline_depth: float = 0.05
    diagonal_depth: float = 0.05
    mid_vert_depth: float = 0.05
    side_vert_depth: float = 0.05
    top_depth: float = 0.05
    bottom_depth: float = 0.05
    
    
    E: float = 200e9
    sigma_compression: float = 250e6
    sigma_tension: float = 400e6
    density: float = 7850.0  

    def __post_init__(self):
        
        if self.material is not None:
            self.E = self.material.E
            self.sigma_compression = self.material.sigma_compression
            self.sigma_tension = self.material.sigma_tension
            self.density = self.material.density

def random_initial_params(bounds: Dict[str, Tuple[float, float]] = None, material: Optional[Material] = None, seed: Optional[int] = None) -> BridgeDesignParams:
    
    import random
    if seed is not None:
        random.seed(seed)
    
    params = {}
    for field in BridgeDesignParams.__dataclass_fields__.values():

        if field.name == "height":
            
            
            max_height = (params.get('length', inches_to_meters(18.5)) * math.tan(params.get('angle'))) / 2
            if bounds and 'height' in bounds:
                min_val, max_val = bounds['height']
                max_val = min(max_val, max_height)

                if min_val >= max_val:
                    print(f"It seems {math.degrees(params.get("angle"))} degrees is an impossible angle given the minimum height provided for the competition constraints.")
                    raise ValueError(f"Invalid bounds for height: min {min_val} >= max {max_val} (max allowed by geometry is {max_height})")

                params['height'] = random.uniform(min_val, max_val)
            else:
                params['height'] = random.uniform(inches_to_meters(4.0), max_height)

            continue
            
        name = field.name
        if bounds and name in bounds:
            min_val, max_val = bounds[name]
            params[name] = random.uniform(min_val, max_val)
        else:
            params[name] = field.default
    
    if material is not None:
        params['material'] = material
    
    return BridgeDesignParams(**params)



class BridgeOptimizer:
    
    def __init__(self,
                 initial_params: BridgeDesignParams,
                 fixed_params: Dict[str, bool] = None,
                 material: Optional[object] = None,
                 param_bounds: Dict[str, Tuple[float, float]] = None,
                 learning_rate: float = 0.1,
                 device: Optional[str] = None,
                 ratio_mode: Optional[str] = 'max_load_per_weight',
                 fixed_cost=0.0,
                 verbose: bool = True):
        self.device = device or 'cpu'
        self.learning_rate = learning_rate
        self.initial_params = initial_params
        
        self.initial_params_dict = asdict(initial_params)

        
        
        
        
        if material is not None:
            
            mat_vals = {}
            for attr in ('E', 'sigma_compression', 'sigma_tension'):
                if hasattr(material, attr):
                    mat_vals[attr] = getattr(material, attr)

            
            temp_init = dict(self.initial_params_dict)
            temp_init.update(mat_vals)
            self.initial_params_dict = temp_init

        if fixed_params is None:
            fixed_params = {
                'E': True,
                'sigma_compression': True,
                'sigma_tension': True,
            }
        self.fixed_params = fixed_params if fixed_params != 'default' else {
                'E': True,
                'sigma_compression': True,
                'sigma_tension': True,
                'density': True,
                'length': True,  
                'material': True,  
            }
        
        
        self.param_bounds = param_bounds or {}
        
        
        self.trainable_params = {}
        for param_name, param_value in self.initial_params_dict.items():
            is_fixed = self.fixed_params.get(param_name, False)
            requires_grad = not is_fixed

            
            if isinstance(param_value, (float, int)):
                tensor = torch.tensor(param_value, dtype=DTYPE, device=self.device, requires_grad=requires_grad)
                self.trainable_params[param_name] = tensor
        
        
        trainable_tensors = [v for k, v in self.trainable_params.items() 
                             if not self.fixed_params.get(k, False)]
        self.optimizer = torch.optim.Adam(trainable_tensors, lr=learning_rate)
        
        
        self.apply_bounds()
        
        
        self.loss_history = []
        self.critical_load_history = []
        self.param_history = {name: [] for name in self.trainable_params}

        
        self.best_objective_value = float('inf')  
        self.best_params: Dict[str, float] = {}
        self.best_iteration = -1

        
        
        
        
        assert (ratio_mode is None) or (ratio_mode in ('weight_per_load', 'max_load_per_weight')), "invalid ratio_mode"
        self.ratio_mode = ratio_mode

        self.density = initial_params.density
        self.fixed_cost = fixed_cost
        self.verbose = verbose
        
        
        if fixed_cost != 0.0 and verbose:
            print(f"WARNING: Fixed cost per plane is {fixed_cost:.4f} N (will be added to weight in loss function)")
        
        
        self._last_summary_time = 0.0
    
    def loss_function(self) -> torch.Tensor:
        max_load, _ = max_load_torch(
            angle=self.trainable_params['angle'],
            height=self.trainable_params['height'],
            length=self.trainable_params['length'],
            incline_thickness=self.trainable_params['incline_thickness'],
            diagonal_thickness=self.trainable_params['diagonal_thickness'],
            mid_vert_thickness=self.trainable_params['mid_vert_thickness'],
            side_vert_thickness=self.trainable_params['side_vert_thickness'],
            top_thickness=self.trainable_params['top_thickness'],
            bottom_thickness=self.trainable_params['bottom_thickness'],
            incline_depth=self.trainable_params['incline_depth'],
            diagonal_depth=self.trainable_params['diagonal_depth'],
            mid_vert_depth=self.trainable_params['mid_vert_depth'],
            side_vert_depth=self.trainable_params['side_vert_depth'],
            top_depth=self.trainable_params['top_depth'],
            bottom_depth=self.trainable_params['bottom_depth'],
            E=self.trainable_params['E'],
            sigma_compression=self.trainable_params['sigma_compression'],
            sigma_tension=self.trainable_params['sigma_tension'],
            softness=1e-6,
            dtype=DTYPE,
            device=self.device,
        )
        
        
        if self.ratio_mode is None:
            return -max_load

        
        
        
        
        weight = compute_weight_torch(
            incline_thickness=self.trainable_params['incline_thickness'],
            diagonal_thickness=self.trainable_params['diagonal_thickness'],
            mid_vert_thickness=self.trainable_params['mid_vert_thickness'],
            side_vert_thickness=self.trainable_params['side_vert_thickness'],
            top_thickness=self.trainable_params['top_thickness'],
            bottom_thickness=self.trainable_params['bottom_thickness'],
            incline_depth=self.trainable_params['incline_depth'],
            diagonal_depth=self.trainable_params['diagonal_depth'],
            mid_vert_depth=self.trainable_params['mid_vert_depth'],
            side_vert_depth=self.trainable_params['side_vert_depth'],
            top_depth=self.trainable_params['top_depth'],
            bottom_depth=self.trainable_params['bottom_depth'],
            length=self.trainable_params['length'],
            height=self.trainable_params['height'],
            angle=self.trainable_params['angle'],
            density=self.density,
            dtype=DTYPE,
            device=self.device,
        )

        
        weight += torch.tensor(self.fixed_cost, dtype=DTYPE, device=self.device)

        
        max_load_clamped = torch.clamp(max_load, min=1e-6)
        weight_clamped = torch.clamp(weight, min=1e-12)
        if self.ratio_mode == 'weight_per_load':
            return weight_clamped / max_load_clamped
        else:
            
            return -max_load_clamped / weight_clamped
    
    def apply_bounds(self):
        
        for param_name, (min_val, max_val) in self.param_bounds.items():
            
            if param_name in self.trainable_params and not self.fixed_params.get(param_name, False):
                with torch.no_grad():
                    self.trainable_params[param_name].clamp_(min_val, max_val)
    
    def train_step(self, iteration: int) -> float:
        self.optimizer.zero_grad()
        loss = self.loss_function()
        
        
        loss_val = loss.item()
        if loss_val < self.best_objective_value:
            self.best_objective_value = loss_val
            self.best_iteration = iteration
            
            self.best_params = {}
            for name, tensor in self.trainable_params.items():
                self.best_params[name] = tensor.detach().item()
            
            for name, value in self.initial_params_dict.items():
                if name not in self.best_params:
                    self.best_params[name] = value
        
        loss.backward()
        self.optimizer.step()
        self.apply_bounds()
        
        
        
        
        if self.ratio_mode is None:
            crit_load_val = -loss_val
        else:
            crit_load_val = loss_val
        
        self.loss_history.append(loss_val)
        self.critical_load_history.append(crit_load_val)
        
        for param_name, param_tensor in self.trainable_params.items():
            self.param_history[param_name].append(param_tensor.item())
        
        return loss_val
    
    def train(self, num_iterations: int, verbose: bool = True, log_interval: int = 10) -> Tuple[Dict, Dict]:
        for iteration in range(num_iterations):
            loss = self.train_step(iteration)
            
            if verbose and (iteration + 1) % log_interval == 0:
                if self.ratio_mode is None:
                    crit_load = -loss
                    print(f"[Iter {iteration + 1:4d}] Loss: {loss:10.2f} | Critical Load: {crit_load:.2f} N")
                else:
                    if self.ratio_mode == 'weight_per_load':
                        print(f"[Iter {iteration + 1:3d}] Loss (weight/load): {loss:4.2f}")
                    else:
                        print(f"[Iter {iteration + 1:3d}] load/weight: {-loss:4.2f}")
        
        
        '''
            if verbose:
                final_loss = self.loss_history[-1]
                
                now = time.time()
                if now - self._last_summary_time < 0.5:
                    
                    pass
                else:
                    self._last_summary_time = now
                    if self.ratio_mode is None:
                        final_crit_load = -final_loss
                        initial_crit_load = -self.loss_history[0] if self.loss_history else 0
                        improvement_pct = ((final_crit_load - initial_crit_load) / abs(initial_crit_load) * 100) if initial_crit_load != 0 else 0
                        print(f"\nTraining complete!")
                        print(f"  Initial critical load: {initial_crit_load:.2f} N")
                        print(f"  Final critical load:   {final_crit_load:.2f} N")
                        print(f"  Improvement: {improvement_pct:.2f}%")
                    else:
                        print(f"\nTraining complete!")
                        if self.ratio_mode == 'weight_per_load':
                            print(f"  Final weight/load ratio: {final_loss:.2f}")
                            print(f"  (Lower is better = higher strength-to-weight)")
                        else:
                            print(f"  Final -load/weight (minimized): {final_loss:.2f}")
                            print(f"  (More negative is better = higher load-to-weight)")
        '''

        
        if verbose and self.best_params:
            best_iter_display = self.best_iteration + 1
            if self.ratio_mode == 'max_load_per_weight':
                best_load_weight = -self.best_objective_value
                print(f"\n  Best load/weight ratio: {best_load_weight:.2f} (achieved at iteration {best_iter_display})")
            elif self.ratio_mode == 'weight_per_load':
                print(f"\n  Best weight/load ratio: {self.best_objective_value:.2f} (achieved at iteration {best_iter_display})")
            else:
                best_crit_load = -self.best_objective_value
                print(f"\n  Best critical load: {best_crit_load:.2f} N (achieved at iteration {best_iter_display})")
        
        return self.initial_params_dict, self.best_params
    
    def get_optimization_history(self) -> Dict:
        
        return {
            'loss_history': self.loss_history,
            'critical_load_history': self.critical_load_history,
            'param_history': self.param_history,
        }
    
    def get_param_deltas(self, initial: Dict, final: Dict) -> Dict[str, float]:
        
        deltas = {}
        for key in initial:
            if key in final:
                
                if isinstance(initial[key], (int, float)) and isinstance(final[key], (int, float)):
                    deltas[key] = final[key] - initial[key]
        return deltas



def make_and_train_pratt(seed: int = 42, iterations: int = 200, lr: float = 0.01, verbose: bool = True, material=HighDensityBalsaWood(), log_interval: int = 20, flat_factor: float = 0.5, fixed_cost: float = 0.0) -> Tuple[Dict, Dict]:
    def vprint(*args, **kwargs):
        if verbose:
            print(*args, **kwargs)

    
    vprint("=" * 70); vprint("Bridge Design Optimizer - Example Run"); vprint("=" * 70)
    
    
    min = inches_to_meters(0.1)
    param_bounds = {
        'length': (inches_to_meters(18.5), inches_to_meters(18.5)),  
        'angle': (math.atan(inches_to_meters(10.0)/(inches_to_meters(18.5)/2)), math.radians(85)), 
                                                                                                   
                                                                                                   
        'height': (inches_to_meters(4.0), inches_to_meters(10.0)), 
        'incline_thickness': (min, inches_to_meters(0.5)), 
        'diagonal_thickness': (min, inches_to_meters(0.5)),
        'mid_vert_thickness': (min, inches_to_meters(0.5)),
        'side_vert_thickness': (min, inches_to_meters(0.5)),
        'top_thickness': (min, inches_to_meters(0.5)),
        'bottom_thickness': (min, inches_to_meters(0.5)),
        'incline_depth': (min, inches_to_meters(0.5)),
        'diagonal_depth': (min, inches_to_meters(0.5)),
        'mid_vert_depth': (min, inches_to_meters(0.5)),
        'side_vert_depth': (min, inches_to_meters(0.5)),
        'top_depth': (min, inches_to_meters(0.5)),
        'bottom_depth': (min, inches_to_meters(0.5)),
    }

    
    initial = random_initial_params(bounds=param_bounds, material=material, seed=seed)
    
    
    optimizer = BridgeOptimizer(
        initial_params=initial,
        param_bounds=param_bounds,
        learning_rate=lr,
        
        fixed_cost=fixed_cost,
        verbose=verbose,
    )
    
    
    init_params, final_params = optimizer.train(num_iterations=iterations, verbose=verbose, log_interval=log_interval)
    
    
    vprint("\n" + "=" * 70); vprint("Parameter Changes (Initial -> Final)"); vprint("=" * 70)
    deltas = optimizer.get_param_deltas(init_params, final_params)
    for param_name in sorted(deltas.keys()):
        if deltas[param_name] != 0:  
            init_val = init_params[param_name]
            final_val = final_params[param_name]
            delta = deltas[param_name]
            pct_change = (delta / init_val * 100) if init_val != 0 else 0
            vprint(f"  {param_name:25s}: {init_val:4.2e} → {final_val:4.2e} (Δ {delta:+.3e}, {pct_change:+.2f}%)")

    return init_params, final_params


## bridges_parametric.py

In [ ]:
import math
from math import sin, cos, tan

class Bridge:
    def __init__(self, material=None):
        self.material = material

    class TensionMember:
        def __init__(self, ultimate_tensile_str, cross_sec_area, M_max=0, depth=None, length=None):
            self.ultimate_tensile_str = ultimate_tensile_str
            self.cross_sec_area = cross_sec_area
            
            self.area = self.cross_sec_area
            self.M_max = M_max
            self.depth = depth  
            self.length = length
        
        @property
        def volume(self):
            
            if self.length is not None:
                return self.cross_sec_area * self.length
            return 0.0


        def get_failure_force(self):
            return self.cross_sec_area * self.ultimate_tensile_str

        def combined_stress_failure_ratio(self, F_axial):
            sigma = F_axial / self.cross_sec_area
            if self.M_max and self.depth:
                
                thickness = self.cross_sec_area / self.depth
                I = self.depth * thickness**3 / 12
                c = thickness / 2
                sigma += self.M_max * c / I
            return sigma / self.ultimate_tensile_str
        
        def find_failure_F(self, axial_lambda, moment_lambda=None):
            from scipy.optimize import fsolve

            def failure_equation(F):
                F_axial = axial_lambda(F)
                M_max = moment_lambda(F) if moment_lambda else 0
                sigma = F_axial / self.cross_sec_area
                if M_max and self.depth:
                    
                    
                    thickness = self.cross_sec_area / self.depth
                    I = self.depth * thickness**3 / 12
                    c = thickness / 2
                    
                    sigma += abs(M_max) * c / I
                return sigma / self.ultimate_tensile_str - 1.0

            
            
            initial_guesses = [
                self.get_failure_force(),
                self.get_failure_force() / 2,
                self.get_failure_force() * 2,
                self.get_failure_force() / 10,
            ]
            
            positive_roots = []
            for guess in initial_guesses:
                try:
                    root = fsolve(failure_equation, guess)[0]
                    if root > 1e-6:  
                        positive_roots.append(root)
                except:
                    continue
            
            
            if positive_roots:
                return min(positive_roots)
            else:
                
                return abs(fsolve(failure_equation, self.get_failure_force())[0])
        
        def get_volume(self, len):
            return len * self.cross_sec_area

    class CompressionMember:
        def __init__(self, length, thickness, depth, K, E, sigma_ult):
            self.length = length
            self.thickness = thickness
            self.depth = depth
            self.K = K
            self.E = E
            self.sigma_ult = sigma_ult

            self.cross_sec_area = thickness * depth

            
            self.A = thickness * depth
            
            self.area = self.A
            self.I_in_plane = depth * thickness**3 / 12
            self.I_out_of_plane = thickness * depth**3 / 12
            self.c = thickness / 2

            self.volume = self.length * self.thickness * self.depth

        def max_compressive_force_in_plane(self, is_in_plane=True):
            F_cr = (3.14159265**2) * self.E * (self.I_in_plane if is_in_plane else self.I_out_of_plane) / (self.K * self.length)**2
            return F_cr

        def combined_stress_failure_ratio(self, F_axial, M_max=0, is_in_plane=True):
            sigma = F_axial / self.A + M_max * self.c / (self.I_in_plane if is_in_plane else self.I_out_of_plane)
            return sigma / self.sigma_ult
        
        def find_euler_buckle_F(self, axial_lambda, moment_lambda=None, is_in_plane=True):
            from scipy.optimize import fsolve
            
            
            F_cr_member = self.max_compressive_force_in_plane(is_in_plane)
            
            if moment_lambda is None:
                
                F_euler = F_cr_member / abs(axial_lambda(1.0))
                return F_euler
            
            
            
            
            def buckling_interaction_equation(F):
                F_axial = axial_lambda(F)
                M_max = moment_lambda(F)
                I = self.I_in_plane if is_in_plane else self.I_out_of_plane
                c = self.c
                
                ratio = F_axial / F_cr_member + (M_max * c * self.A) / (F_cr_member * I)
                return ratio - 1.0
            
            
            initial_guess = F_cr_member / (2.0 * abs(axial_lambda(1.0)))
            try:
                F_euler_with_moment = fsolve(buckling_interaction_equation, initial_guess)[0]
            except:
                
                F_euler_with_moment = F_cr_member / abs(axial_lambda(1.0))
            
            return F_euler_with_moment
        
        def find_material_strength_F(self, axial_lambda, moment_lambda=None, is_in_plane=True):
            from scipy.optimize import fsolve
            
            def material_failure_equation(F):
                F_axial = axial_lambda(F)
                M_max = moment_lambda(F) if moment_lambda else 0
                sigma = F_axial / self.A
                if M_max:
                    I = self.I_in_plane if is_in_plane else self.I_out_of_plane
                    c = self.c
                    sigma += M_max * c / I
                return sigma / self.sigma_ult - 1.0

            
            initial_guess = self.A * self.sigma_ult / abs(axial_lambda(1.0))
            try:
                F_material = fsolve(material_failure_equation, initial_guess)[0]
            except:
                F_material = float('inf')
            
            return F_material
        
        def find_buckle_F(self, axial_lambda, moment_lambda=None, is_in_plane=True):
            F_euler = self.find_euler_buckle_F(axial_lambda, moment_lambda, is_in_plane)
            F_material = self.find_material_strength_F(axial_lambda, moment_lambda, is_in_plane)
            return min(F_euler, F_material)


class OnePanelPratt2D(Bridge):
    def __init__(self, angle, height, length,
                 incline_thickness, diagonal_thickness, mid_vert_thickness, side_vert_thickness, top_thickness, bottom_thickness,
                 incline_depth, diagonal_depth, mid_vert_depth, side_vert_depth, top_depth, bottom_depth,
                 E, sigma_compression, sigma_tension, material=None):
        super().__init__(material=material)

        self.angle = angle
        self.height = height
        self.length = length
        self.phi = math.atan(((self.length-2*height/tan(angle))/2)/(height))

        
        self.members = {
            "incline": self.CompressionMember(length=height/sin(angle), 
                                              thickness=incline_thickness,
                                              depth=incline_depth, 
                                              K=1, 
                                              E=E, 
                                              sigma_ult=sigma_compression),

            "diagonal": self.TensionMember(ultimate_tensile_str=sigma_tension,
                                           cross_sec_area=diagonal_thickness*diagonal_depth,
                                           depth=diagonal_depth,
                                           length=height/cos(self.phi)),

            "top_chord": self.CompressionMember(length=length-(2*height/tan(angle)), 
                                                thickness=top_thickness,
                                                depth=top_depth, 
                                                K=1, 
                                                E=E, 
                                                sigma_ult=sigma_compression),

            "bottom_chord": self.TensionMember(ultimate_tensile_str=sigma_tension,
                                               cross_sec_area=bottom_thickness*bottom_depth,
                                               depth=bottom_depth,
                                               length=length),

            "mid_vert": self.CompressionMember(length=height, 
                                               thickness=mid_vert_thickness,
                                               depth=mid_vert_depth, 
                                               K=1, 
                                               E=E, 
                                               sigma_ult=sigma_compression),

            "side_vert": self.CompressionMember(length=height,
                                                thickness=side_vert_thickness,
                                                depth=side_vert_depth,
                                                K=1,
                                                E=E,
                                                sigma_ult=sigma_compression
            )
        }

    def get_axial_forces_from_F_dict(self):
        axial_forces = {}
        axial_forces["incline"] = lambda F: F / (2 * sin(self.angle))
        axial_forces["diagonal"] = lambda F: F / (6 * cos(self.phi))
        axial_forces["top_chord"] = lambda F: axial_forces["incline"](F)*cos(self.angle) + axial_forces["diagonal"](F)*sin(self.phi)
        axial_forces["bottom_chord"] = axial_forces["top_chord"]

        total_vertical_membs_area = 2*self.members["side_vert"].area + self.members["mid_vert"].area

        total_force_covered_by_vert_membs = lambda F: F - axial_forces["incline"](F)*sin(self.angle)*2 + axial_forces["diagonal"](F)*cos(self.phi)*2

        axial_forces["mid_vert"] = lambda F_axial: self.members["mid_vert"].area / total_vertical_membs_area * total_force_covered_by_vert_membs(F_axial)
        axial_forces["side_vert"] = lambda F_axial: self.members["side_vert"].area / total_vertical_membs_area * total_force_covered_by_vert_membs(F_axial)
        return axial_forces
    
    def get_required_F_from_axial_forces_dict(self):
        
        load_formulas = {}
        load_formulas["incline"] = lambda F_axial: 2 * F_axial * sin(self.angle)
        load_formulas["diagonal"] = lambda F_axial: 6 * F_axial * cos(self.phi)
        
        load_formulas["top_chord"] = lambda F_axial: F_axial / (cos(self.angle) / (2 * sin(self.angle)) + sin(self.phi) / (6 * cos(self.phi)))
        load_formulas["bottom_chord"] = load_formulas["top_chord"]
        
        
        
        
        
        
        
        total_vertical_membs_area = 2 * self.members["side_vert"].area + self.members["mid_vert"].area

        load_formulas["mid_vert"] = lambda F_axial: 3 * F_axial * total_vertical_membs_area / self.members["mid_vert"].area
        load_formulas["side_vert"] = lambda F_axial: 3 * F_axial * total_vertical_membs_area / self.members["side_vert"].area

        
        return load_formulas
    
    def get_moments_from_F_dict(self):
        moments = {}
        
        moments["incline"] = [lambda F: 0.0]
        moments["diagonal"] = [lambda F: 0.0]

        axial_dict = self.get_axial_forces_from_F_dict()

        side_of_top_vertical_force = lambda F: axial_dict["incline"](F)*sin(self.angle) + axial_dict["side_vert"](F) - axial_dict["diagonal"](F)*cos(self.phi) - F/3
        moment_at_center_top = lambda F: side_of_top_vertical_force(F) * (self.length/2 - self.height/tan(self.angle))
        moment_at_end_top = lambda F: side_of_top_vertical_force(F) * (self.length - 2*self.height/tan(self.angle)) + (axial_dict["mid_vert"](F)-F/3) * (self.length/2 - self.height/tan(self.angle))
        assert math.isclose(side_of_top_vertical_force(1.0)*2+axial_dict["mid_vert"](1.0)-1.0/3, 0.0, rel_tol=1e-9, abs_tol=1e-12), f"Debug: vertical forces on top chord do not sum to zero! In fact they sum to {side_of_top_vertical_force(1.0)*2+axial_dict['mid_vert'](1.0)-1.0/3}"
        moments["top_chord"] = [moment_at_center_top, moment_at_end_top]

        side_of_bottom_vertical_force = lambda F: F/2 - axial_dict["incline"](F)*sin(self.angle)
        intermediate_bottom_force = lambda F: -axial_dict["side_vert"](F)
        center_bottom_force = lambda F: 2*axial_dict["diagonal"](F)*cos(self.phi) - axial_dict["mid_vert"](F)
        moment1_bottom = lambda F: side_of_bottom_vertical_force(F) * self.height/tan(self.angle)
        moment2_bottom = lambda F: side_of_bottom_vertical_force(F)*self.length/2 + intermediate_bottom_force(F) * (self.length/2 - self.height/tan(self.angle))
        moment3_bottom = lambda F: side_of_bottom_vertical_force(F)*(self.length-self.height/tan(self.angle)) + intermediate_bottom_force(F)*(self.length - 2*self.height/tan(self.angle)) + center_bottom_force(F)*(self.length/2-self.height/tan(self.angle))
        moment4_bottom = lambda F: side_of_bottom_vertical_force(F)*self.length + intermediate_bottom_force(F)*(self.length - self.height/tan(self.angle)) + center_bottom_force(F)*(self.length/2) + intermediate_bottom_force(F)*(self.height/tan(self.angle))
        assert math.isclose(side_of_bottom_vertical_force(1.0)*2 + intermediate_bottom_force(1.0)*2 + center_bottom_force(1.0), 0.0, rel_tol=1e-9, abs_tol=1e-12), f"Debug: vertical forces on bottom chord do not sum to zero! In fact they sum to {side_of_bottom_vertical_force(1.0)*2+intermediate_bottom_force(1.0)*2+center_bottom_force(1.0)}. The entire expression is {side_of_bottom_vertical_force(1.0)}*2 + {intermediate_bottom_force(1.0)}*2 + {center_bottom_force(1.0)}"
        
        moments["bottom_chord"] = [moment1_bottom, moment2_bottom, moment3_bottom, moment4_bottom]

        moments["mid_vert"] = [lambda F: 0.0]
        moments["side_vert"] = [lambda F: 0.0]
        return moments
    
    def get_required_F_from_moments_dict(self):
        from scipy.optimize import fsolve
        load_from_moments = {}
        
        
        moment_fns_dict = self.get_moments_from_F_dict()
        
        
        def is_structurally_zero(moment_fn):
            
            test_values = [0.1, 1.0, 10.0, 100.0, 1000.0]
            for F in test_values:
                if abs(moment_fn(F)) > 1e-9:
                    return False
            return True
        
        
        def initial_guess(M):
            
            return 1.0 if abs(M) < 1e-6 else M / 100.0
        
        
        def create_inverse_fn(moment_fn):
            
            if is_structurally_zero(moment_fn):
                return None
            else:
                return lambda M: fsolve(lambda F: moment_fn(F) - M, initial_guess(M))[0]
        
        
        top_moment_fns = moment_fns_dict["top_chord"]
        load_from_moments["top_chord"] = create_inverse_fn(top_moment_fns[0])
        
            
            
        
        
        
        bottom_moment_fns = moment_fns_dict["bottom_chord"]
        load_from_moments["bottom_chord"] = create_inverse_fn(bottom_moment_fns[1])
        
            
            
            
            
        
        
        
        load_from_moments["incline"] = None
        load_from_moments["diagonal"] = None
        load_from_moments["mid_vert"] = None
        load_from_moments["side_vert"] = None
        return load_from_moments
    
    def get_failure_mode_dict(self):
        req_F_from_axial = self.get_required_F_from_axial_forces_dict()
        req_F_from_moments = self.get_required_F_from_moments_dict()

        f_mode_dict = {}

        for name, member in self.members.items():
            
            axial_forward = self.get_axial_forces_from_F_dict()
            moments_forward = self.get_moments_from_F_dict()

            f_mode_dict = {}

            for name, member in self.members.items():
                
                axial_lambda = axial_forward.get(name)

                
                moment_lambda = None
                mf = moments_forward.get(name)
                if mf:
                    
                    if isinstance(mf, list):
                        
                        for fn in mf:
                            try:
                                if abs(fn(1.0)) > 1e-12:
                                    moment_lambda = fn
                                    break
                            except Exception:
                                continue
                    elif callable(mf):
                        moment_lambda = mf

                if isinstance(member, self.TensionMember):
                    f_mode_dict[f"{name}_rupture"] = member.find_failure_F(
                        axial_lambda=axial_lambda,
                        moment_lambda=moment_lambda,  
                    )
                elif isinstance(member, self.CompressionMember):
                    
                    f_mode_dict[f"{name}_buckle"] = member.find_euler_buckle_F(
                        axial_lambda=axial_lambda,
                        moment_lambda=moment_lambda,
                        is_in_plane=True,
                    )
                    f_mode_dict[f"{name}_buckle_out_of_plane"] = member.find_euler_buckle_F(
                        axial_lambda=axial_lambda,
                        moment_lambda=moment_lambda,
                        is_in_plane=False,
                    )
                    f_mode_dict[f"{name}_combined_stress"] = member.find_material_strength_F(
                        axial_lambda=axial_lambda,
                        moment_lambda=moment_lambda,
                        is_in_plane=True,  
                    )
                else:
                    raise ValueError(f"Unknown member type for {name}")

            f_mode_dict["torsion_failure"] = None  

            return f_mode_dict
        
    def get_total_volume(self):
        total_volume = 0.0
        total_volume += 2*self.members["incline"].volume
        total_volume += 2*self.members["diagonal"].volume
        total_volume += self.members["top_chord"].volume
        total_volume += self.members["bottom_chord"].volume
        total_volume += self.members["mid_vert"].volume
        total_volume += 2*self.members["side_vert"].volume
        return total_volume


## pratt_visualiser.py

In [ ]:
import turtle
from math import sin, cos, tan
import numpy as np
from PIL import Image

class PrattVisualiser:

    @staticmethod
    def visualise(bridge: OnePanelPratt2D) -> np.ndarray:
        
        turtle.TurtleScreen._RUNNING = True
        turtle._Screen._root = None
        turtle._Screen._canvas = None


        try:
            screen = turtle.getscreen()
            screen.clearscreen()
        except turtle.TurtleGraphicsError:
            screen = turtle.Screen()
        screen.title("Pratt Bridge Visualiser")
        screen.setup(width=800, height=600)
        root = None  
        

        
        screen.setworldcoordinates(-1, -1, bridge.length + 1, bridge.length + 1)
        
        
        t = turtle.RawTurtle(screen)
        t.speed(0)
        t.hideturtle()
        t.pensize(2)
        t.color("black")
        t.penup()
        t.goto(0, 0)
        t.pendown()

        
        t.forward(bridge.length)
        t.left(180 - bridge.angle * 180 / 3.14159)
        t.forward(bridge.height/sin(bridge.angle))
        t.setheading(180)
        t.forward(bridge.length-2*bridge.height/tan(bridge.angle))
        t.left(bridge.angle * 180 / 3.14159)
        t.forward(bridge.height/sin(bridge.angle))
        t.penup()
        t.backward(bridge.height/sin(bridge.angle))
        t.setheading(270)
        t.pendown()
        t.forward(bridge.height)
        t.penup()
        t.backward(bridge.height)
        t.pendown()
        t.left(bridge.phi * 180 / 3.14159)
        t.forward(bridge.height/cos(bridge.phi))
        t.setheading(90)
        t.forward(bridge.height)
        t.penup()
        t.backward(bridge.height)
        t.pendown()
        t.setheading(0)
        t.left(90 - bridge.phi * 180 / 3.14159)
        t.forward(bridge.height/cos(bridge.phi))
        t.setheading(270)
        t.forward(bridge.height)

        
        canvas = screen.getcanvas()
        canvas.postscript(file="turtle_output.ps")
        img = Image.open("turtle_output.ps")
        img_gray = img.convert("L")
        arr = np.array(img_gray)

        
        turtle.TurtleScreen._RUNNING = True
        turtle._Screen._root = None
        turtle._Screen._canvas = None

        return arr

    def __init__(self):
        pass

    def shutdown(self):
        
        try:
            turtle.bye()
        except:
            pass
        turtle.TurtleScreen._RUNNING = True
        turtle._Screen._root = None
        turtle._Screen._canvas = None


## post_process_parameters_perfectly.py

In [ ]:

import json
import os
from pathlib import Path
import numpy as np
import math


def load_trial_data(trials_dir='trials'):
    
    trials_path = Path(trials_dir)
    json_files = sorted(trials_path.glob('final_pratt_bridge*.json'))
    
    if not json_files:
        raise FileNotFoundError(f"No trial JSON files found in {trials_dir}/")
    
    print(f"Loading {len(json_files)} trial files...")
    
    
    trials = []
    for json_file in json_files:
        with open(json_file, 'r') as f:
            trial_data = json.load(f)
            trials.append(trial_data)
    
    return trials


def compute_statistics(trials):
    
    
    param_keys = list(trials[0].keys())
    
    
    param_values = {key: [] for key in param_keys}
    
    for trial in trials:
        for key in param_keys:
            param_values[key].append(trial[key])
    
    
    means = {}
    std_devs = {}
    
    for key in param_keys:
        values = np.array(param_values[key])
        means[key] = float(np.mean(values))
        std_devs[key] = float(np.std(values, ddof=1))  
    
    return means, std_devs


def convert_units(data, target_unit):
    
    converted = {}
    
    
    m_to_cm = 100.0
    m_to_in = 39.3701
    m_to_ft = 3.28084
    rad_to_deg = 180.0 / math.pi
    
    
    if target_unit == 'cm':
        length_factor = m_to_cm
        unit_name = "centimeters"
    elif target_unit == 'in':
        length_factor = m_to_in
        unit_name = "inches"
    elif target_unit == 'ft':
        length_factor = m_to_ft
        unit_name = "feet"
    else:
        raise ValueError(f"Unknown unit: {target_unit}")
    
    
    distance_params = [
        'height', 'length',
        'incline_thickness', 'incline_depth',
        'diagonal_thickness', 'diagonal_depth',
        'mid_vert_thickness', 'mid_vert_depth',
        'side_vert_thickness', 'side_vert_depth',
        'top_thickness', 'top_depth',
        'bottom_thickness', 'bottom_depth'
    ]
    
    for key, value in data.items():
        if key == 'angle':
            converted[key] = value * rad_to_deg
        elif key in distance_params:
            converted[key] = value * length_factor
        else:
            converted[key] = value  
    
    return converted, unit_name


def save_final_plans(means, std_devs, num_trials, output_file='final_plans.json'):
    
    final_plans = {
        "means": means,
        "standard_deviations": std_devs,
        "metadata": {
            "num_trials": num_trials,
            "description": "Aggregated statistics from optimization trials",
            "units": {
                "angle": "radians",
                "height": "meters",
                "length": "meters",
                "thicknesses": "meters",
                "depths": "meters",
                "E": "Pascals",
                "sigma_compression": "Pascals",
                "sigma_tension": "Pascals"
            }
        }
    }
    
    print(f"Saving results to {output_file}...")
    with open(output_file, 'w') as f:
        json.dump(final_plans, f, indent=2, sort_keys=False)
    
    print(f"✓ Successfully saved to {output_file}")


def save_converted_plans(means, std_devs, num_trials, target_unit):
    
    means_converted, unit_name = convert_units(means, target_unit)
    std_devs_converted, _ = convert_units(std_devs, target_unit)
    
    angle_unit = "degrees"
    
    final_plans = {
        "means": means_converted,
        "standard_deviations": std_devs_converted,
        "metadata": {
            "num_trials": num_trials,
            "description": f"Aggregated statistics from optimization trials (converted to {unit_name})",
            "units": {
                "angle": angle_unit,
                "height": unit_name,
                "length": unit_name,
                "thicknesses": unit_name,
                "depths": unit_name,
                "E": "Pascals",
                "sigma_compression": "Pascals",
                "sigma_tension": "Pascals"
            }
        }
    }
    
    output_file = f"final_plans_{target_unit.upper()}.json"
    print(f"Saving converted results to {output_file}...")
    with open(output_file, 'w') as f:
        json.dump(final_plans, f, indent=2, sort_keys=False)
    
    print(f"✓ Successfully saved to {output_file}")


def print_summary(means, std_devs, display_unit='m'):
    
    print("\n" + "="*80)
    print("OPTIMIZATION RESULTS SUMMARY")
    print("="*80)
    
    
    if display_unit in ['cm', 'in', 'ft']:
        means_display, unit_name = convert_units(means, display_unit)
        std_devs_display, _ = convert_units(std_devs, display_unit)
        angle_unit = "degrees"
        length_unit = display_unit
    else:
        means_display = means
        std_devs_display = std_devs
        unit_name = "meters"
        angle_unit = "radians"
        length_unit = "m"
    
    print(f"\nUnits: distances in [{length_unit}], angle in [{angle_unit}]")
    
    
    geometric_params = [
        'angle', 'height', 'length',
        'incline_thickness', 'incline_depth',
        'diagonal_thickness', 'diagonal_depth',
        'mid_vert_thickness', 'mid_vert_depth',
        'side_vert_thickness', 'side_vert_depth',
        'top_thickness', 'top_depth',
        'bottom_thickness', 'bottom_depth'
    ]
    
    material_params = ['E', 'sigma_compression', 'sigma_tension']
    
    print("\n📐 GEOMETRIC PARAMETERS:")
    print(f"{'Parameter':<25} {'Mean':>15} {'Std Dev':>15} {'CV %':>10}")
    print("-"*68)
    for param in geometric_params:
        if param in means_display:
            mean = means_display[param]
            std = std_devs_display[param]
            
            cv = (std_devs[param] / means[param] * 100) if means[param] != 0 else 0
            print(f"{param:<25} {mean:15.6f} {std:15.6f} {cv:9.2f}%")
    
    print("\n🔧 MATERIAL PARAMETERS:")
    print(f"{'Parameter':<25} {'Mean':>15} {'Std Dev':>15} {'CV %':>10}")
    print("-"*68)
    for param in material_params:
        if param in means_display:
            mean = means_display[param]
            std = std_devs_display[param]
            cv = (std / mean * 100) if mean != 0 else 0
            print(f"{param:<25} {mean:15.2e} {std:15.2e} {cv:9.2f}%")
    
    print("\n" + "="*80)


def main(display_unit='m'):
    print("="*80)
    print("BRIDGE OPTIMIZATION POST-PROCESSING")
    print("="*80)
    
    
    trials = load_trial_data('trials')
    num_trials = len(trials)
    
    
    means, std_devs = compute_statistics(trials)
    
    
    save_final_plans(means, std_devs, num_trials, 'final_plans.json')
    
    
    save_converted_plans(means, std_devs, num_trials, 'cm')
    save_converted_plans(means, std_devs, num_trials, 'in')
    
    
    print_summary(means, std_devs, display_unit)
    
    print("\n✓ Post-processing complete!")
    print(f"✓ Processed {num_trials} trials")
    print(f"✓ Results saved to:")
    print(f"   - final_plans.json (SI units: meters, radians)")
    print(f"   - final_plans_CM.json (centimeters, degrees)")
    print(f"   - final_plans_IN.json (inches, degrees)")



## pratt_analyse_v2.py

In [ ]:



import math
import os
import sys


sys.path.insert(0, os.path.abspath(os.path.dirname(__file__)))


def make_and_report(bridge: OnePanelPratt2D = None, density: float = None, special_message: str = None, unit_system: str = 'metric'):
    b = OnePanelPratt2D(
        angle=math.radians(30),
        height=2,
        length=10,
        incline_thickness=0.02,
        diagonal_thickness=0.02,
        mid_vert_thickness=0.02,
        side_vert_thickness=0.02,
        top_thickness=0.02,
        bottom_thickness=0.02,
        incline_depth=0.05,
        diagonal_depth=0.05,
        mid_vert_depth=0.05,
        side_vert_depth=0.05,
        top_depth=0.05,
        bottom_depth=0.05,
        E=200e9,
        sigma_compression=250e6,
        sigma_tension=400e6,
    ) if bridge is None else bridge

    print("\n" + "="*60)
    print("  BRIDGE FAILURE ANALYSIS" if special_message == None else f"  {special_message}")
    print("="*60 + "\n")

    
    failure_modes = b.get_failure_mode_dict()
    
    
    member_results = {}
    member_types = {}
    
    for name, member in b.members.items():
        member_types[name] = "Tension" if isinstance(member, b.TensionMember) else "Compression"
        
        
        member_failures = []
        for mode_name, F_failure in failure_modes.items():
            if F_failure is not None and mode_name != 'torsion_failure':
                if mode_name.startswith(name + '_') or mode_name == name:
                    member_failures.append(F_failure)
        
        if member_failures:
            member_results[name] = min(member_failures)
        else:
            member_results[name] = None
    
    
    for name in b.members.keys():
        applied_load = member_results.get(name)
        member_type = member_types[name]
        if applied_load is not None:
            print(f"  {name:.<20} {member_type:12} {applied_load:>10.2f} N")
        else:
            print(f"  {name:.<20} {member_type:12} {'ERROR':>10}")
    
    if member_results:
        valid_loads = [v for v in member_results.values() if v is not None]
        if valid_loads:
            min_load = min(valid_loads)
            governing = [k for k, v in member_results.items() if v == min_load][0]
            print("\n" + "-"*60)
            print(f"  Governing Member: {governing}")
            print(f"  Critical Load:    {min_load:.2f} N")
            
            
            if density is not None:
                volume = b.get_total_volume()
                mass = volume * density
                material_weight = mass * 9.81  
                weight = material_weight + fixed_cost  
                load_to_weight = min_load / weight if weight > 0 else float('inf')
                
                if unit_system == 'imperial':
                    volume_display = f"{volume / (0.0254 ** 3):.2f} in³"
                    mass_display = f"{mass * 35.274:.2f} oz"
                else:  
                    volume_display = f"{volume*1e6:.2f} cm³"
                    mass_display = f"{mass*1000:.2f} g"
                
                print(f"\n  Volume:           {volume_display}")
                print(f"  Density:          {density:.2f} kg/m³")
                print(f"  Mass:             {mass_display}")
                if fixed_cost != 0.0:
                    print(f"  Material Weight:        {material_weight:.2f} N")
                    print(f"  Connector Cost/Plane:   {fixed_cost:.2f} N")
                    print(f"  Total Weight:           {weight:.2f} N")
                else:
                    print(f"  Weight:           {weight:.2f} N")
                print(f"  Load/Weight:      {load_to_weight:.2f}")
            
            print("="*60 + "\n")
            if b.material:
                print(f"  Material:         {b.material}")
            else:
                print("  Material:         None")

    return member_results



## main.py

In [ ]:
import json

_MATERIALS_MAIN = [ConservativeBalsaWood, HighDensityBalsaWood, LowDensityBalsaWood, OchromaWood, OchromaWithEpoxy]
_MATERIAL_MAIN = 4 

def fully_abstracted_pratt_workflow(seed: int = 42, verbose: bool = True, terminate_turtle: bool = True, flat_factor: float=0):

    

    
    material = _MATERIALS_MAIN[_MATERIAL_MAIN]()
    initial_params, final_params = make_and_train_pratt(seed=seed, iterations=7000, lr=0.001, verbose=verbose, material=material, flat_factor=flat_factor, fixed_cost=fixed_cost)

    
    density = material.density
    
    
    for param in ['material', 'density']:
        if param in initial_params:
            del initial_params[param]
        if param in final_params:
            del final_params[param]

    
    vis = PrattVisualiser()
    bridge_image_init = vis.visualise(OnePanelPratt2D(**initial_params)) 
    bridge_image_final = vis.visualise(OnePanelPratt2D(**final_params))

    
    import numpy as np
    from PIL import Image
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))

    axs[0].imshow(np.array(bridge_image_init), cmap='gray')
    axs[0].set_title('Initial Bridge Design')
    axs[0].axis('off')
    axs[1].imshow(np.array(bridge_image_final), cmap='gray')
    axs[1].set_title('Final Bridge Design')
    axs[1].axis('off')
    plt.savefig(f"trials/pratt_bridge_comparison{seed}.png")
    if verbose:
        print(f"Saved bridge comparison image as 'trials/pratt_bridge_comparison{seed}.png'")

    
        

    
    if True:
        print('\n')
        
        final_bridge = OnePanelPratt2D(**final_params)
        with open(f'trials/final_pratt_bridge{seed}.json', 'w') as f:
            f.write(final_params.__repr__().replace("'", '"'))
        if verbose:
            print(f"Saved final bridge design to 'trials/final_pratt_bridge{seed}.json'")

        def dict_pretty_print(d: dict):
            import json
            from collections import OrderedDict

            def sort_key(item):
                key, value = item
                if value is None:
                    return (1, key)  
                return (0, value)  

            ordered = OrderedDict(sorted(d.items(), key=sort_key))
            print(json.dumps(ordered, indent=4))
            
        final_bridge = OnePanelPratt2D(**final_params)
        dict_to_be_printed = final_bridge.get_failure_mode_dict()
        
        if density is not None:
            material_weight = final_bridge.get_total_volume() * density * 9.81  
            dict_to_be_printed["weight_N"] = material_weight + fixed_cost
            if fixed_cost != 0.0:
                dict_to_be_printed["material_weight_N"] = material_weight
                dict_to_be_printed["fixed_cost_per_plane_N"] = fixed_cost
        
        if verbose:
            dict_pretty_print(dict_to_be_printed)

        if terminate_turtle:
            vis.shutdown()




